In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
from IPython.display import display, Markdown

# Configuration initiale pour Jupyter Notebook
try:
    plt.style.use('seaborn-v0_8')  # Style compatible avec les nouvelles versions
except:
    plt.style.use('ggplot')  # Fallback si seaborn-v0_8 n'est pas disponible
    
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.family'] = 'DejaVu Sans'  # Police avec meilleure couverture Unicode
warnings.filterwarnings('ignore')  # Supprime les warnings non critiques
sns.set_style("whitegrid")  # Style Seaborn complémentaire

def analyze_dataset(file_path):
    try:
        # Chargement du DataSet
        print(f"\n\033[1mChargement du fichier: {file_path}\033[0m")
        try:
            df = pd.read_csv(file_path)
        except FileNotFoundError:
            raise FileNotFoundError(f"Le fichier '{file_path}' est introuvable. Vérifiez le chemin.")
        
        # Nettoyage initial
        df = clean_data(df)
        
        # Analyse basique
        basic_analysis(df)
        
        # Analyse par type de données
        analyze_by_dtype(df)
        
    except Exception as e:
        print(f"\n\033[1;31mERREUR: {str(e)}\033[0m")

def clean_data(df):
    """Nettoyage initial des données"""
    # Suppression des colonnes vides
    initial_cols = len(df.columns)
    df = df.dropna(axis=1, how='all')
    if len(df.columns) < initial_cols:
        print(f"\n\033[33m{initial_cols - len(df.columns)} colonnes vides supprimées\033[0m")
    
    # Conversion des types
    for col in df.columns:
        # Conversion en numérique
        converted = pd.to_numeric(df[col], errors='ignore')
        if not converted.equals(df[col]):
            df[col] = converted
            print(f"\033[33mColonne '{col}' convertie en numérique\033[0m")
        
        # Conversion des dates
        if df[col].dtype == 'object':
            try:
                df[col] = pd.to_datetime(df[col])
                print(f"\033[33mColonne '{col}' convertie en datetime\033[0m")
            except:
                pass
                
    return df

def basic_analysis(df):
    """Analyse basique du DataFrame"""
    print("\n\033[1m=== ANALYSE BASIQUE ===\033[0m")
    print(f"\nShape du dataset: \033[1m{df.shape}\033[0m")
    
    print("\n\033[1mPremières lignes:\033[0m")
    display(df.head().style.set_caption("Aperçu des données"))
    
    print("\n\033[1mInfos sur les types:\033[0m")
    display(pd.DataFrame(df.dtypes, columns=['Type']).style.set_caption("Types de données"))
    
    print("\n\033[1mValeurs manquantes:\033[0m")
    missing = df.isna().sum().to_frame('Valeurs manquantes')
    missing['%'] = (missing['Valeurs manquantes'] / len(df)) * 100
    display(missing.style.format({'%': '{:.1f}%'}).bar(subset=['%'], color='#ff6961'))
    
    print("\n\033[1mDescription statistique:\033[0m")
    try:
        display(df.describe(include='all').style.set_caption("Statistiques descriptives"))
    except:
        display(df.describe().style.set_caption("Statistiques descriptives (numériques uniquement)"))


def analyze_by_dtype(df):
    """Analyse par type de données"""
    print("\n\033[1m=== ANALYSE PAR TYPE DE DONNÉES ===\033[0m")
    
    # Analyse numérique
    numeric_cols = df.select_dtypes(include=['number']).columns
    if len(numeric_cols) > 0:
        analyze_numeric(df[numeric_cols])
    else:
        print("\n\033[33mAucune colonne numérique trouvée.\033[0m")
    
    # Analyse catégorielle
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    if len(categorical_cols) > 0:
        analyze_categorical(df[categorical_cols])
    else:
        print("\n\033[33mAucune colonne catégorielle trouvée.\033[0m")
    
    # Analyse temporelle
    datetime_cols = df.select_dtypes(include=['datetime']).columns
    if len(datetime_cols) > 0:
        analyze_datetime(df[datetime_cols])
    else:
        print("\n\033[33mAucune colonne temporelle trouvée.\033[0m")

def analyze_numeric(df):
    """Analyse des variables numériques"""
    from IPython.display import display, Markdown
    display(Markdown("### Analyse Numérique"))
    
    # Description statistique
    display(df.describe().style.background_gradient(cmap='Blues'))
    
    # Test de normalité
    norm_results = []
    for col in df.columns:
        try:
            stat, p = stats.shapiro(df[col].dropna())
            norm_results.append({
                'Variable': col,
                'W': f"{stat:.3f}",
                'p-value': f"{p:.3f}",
                'Normalité': 'Normal' if p > 0.05 else 'Non-normal'
            })
        except:
            norm_results.append({
                'Variable': col,
                'W': 'N/A',
                'p-value': 'N/A',
                'Normalité': 'Test non applicable'
            })
    
    display(pd.DataFrame(norm_results).style.set_caption("Tests de normalité (Shapiro-Wilk)"))
    
    # Visualisation
    for col in df.columns:
        try:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            
            # Histogramme
            sns.histplot(df[col], kde=True, ax=ax1)
            ax1.set_title(f'Distribution de {col}')
            
            # Boxplot
            sns.boxplot(x=df[col], ax=ax2)
            ax2.set_title(f'Boxplot de {col}')
            
            plt.tight_layout()
            plt.show()
        except:
            print(f"\033[33mImpossible de visualiser {col}\033[0m")
    
    # Corrélations
    if len(df.columns) >= 2:
        try:
            display(Markdown("#### Matrice de Corrélation"))
            corr = df.corr()
            mask = np.triu(np.ones_like(corr, dtype=bool))
            
            plt.figure(figsize=(10, 8))
            sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', fmt=".2f", 
                       center=0, vmin=-1, vmax=1)
            plt.title('Matrice de Corrélation (triangle inférieur)')
            plt.show()
        except:
            print("\033[33mImpossible de calculer la matrice de corrélation\033[0m")

def analyze_categorical(df):
    """Analyse des variables catégorielles"""
    from IPython.display import display, Markdown
    display(Markdown("### Analyse Catégorielle"))
    
    # Analyse des fréquences
    for col in df.columns:
        try:
            display(Markdown(f"#### {col}"))
            
            # Tableau de fréquences
            freq = df[col].value_counts(dropna=False).to_frame('Count')
            freq['%'] = (freq['Count'] / len(df)) * 100
            display(freq.style.format({'%': '{:.1f}%'}).bar(subset=['%'], color='#5cb85c'))
            
            # Visualisation
            plt.figure(figsize=(10, 5))
            
            if df[col].nunique() > 15:
                # Top 15 pour les variables avec beaucoup de catégories
                top15 = df[col].value_counts().nlargest(15)
                sns.barplot(x=top15.index, y=top15.values)
                plt.xticks(rotation=45, ha='right')
            else:
                # Toutes les catégories
                sns.countplot(x=col, data=df, order=df[col].value_counts().index)
                plt.xticks(rotation=45, ha='right')
            
            plt.title(f'Distribution de {col}')
            plt.tight_layout()
            plt.show()
        except:
            print(f"\033[33mImpossible d'analyser {col}\033[0m")

def analyze_datetime(df):
    """Analyse des variables temporelles"""
    from IPython.display import display, Markdown
    display(Markdown("### Analyse Temporelle"))
    
    for col in df.columns:
        try:
            display(Markdown(f"#### {col}"))
            
            # Statistiques de base
            stats_df = pd.DataFrame({
                'Première date': [df[col].min()],
                'Dernière date': [df[col].max()],
                'Période couverte': [df[col].max() - df[col].min()],
                'Valeurs uniques': [df[col].nunique()],
                'Valeurs manquantes': [df[col].isna().sum()]
            })
            display(stats_df.style.set_caption(f"Statistiques de {col}"))
            
            # Visualisation temporelle
            plt.figure(figsize=(12, 6))
            
            if df[col].nunique() > 100:
                # Agrégation par mois si trop de points
                df[col].hist(bins=50)
                plt.title(f'Distribution de {col} (histogramme)')
            else:
                # Comptage par date exacte
                df[col].value_counts().sort_index().plot(kind='bar')
                plt.title(f'Distribution de {col} (comptage par date)')
            
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
        except:
            print(f"\033[33mImpossible d'analyser {col}\033[0m")

# Pour utiliser dans Jupyter Notebook:
analyze_dataset('dataset_v2-doctored.csv')



Chargement du fichier: dataset_v2-doctored.csv

ERREUR: invalid error value specified
